# Search and filter

Combine filters, page through every result, and load the results into pandas.

In [1]:
import itertools

from seqout import SearchParams, connect
from seqout.models.api_models import SearchResults

sq = connect()

## Filters

`search` takes the query text and any number of filters. The query text is
optional when you give at least one filter.

In [2]:
results = sq.search(
    "single cell",
    db="geo",
    organism="Homo sapiens",
    library_strategy=["RNA-Seq"],
    date_from="2021-01-01",
    date_to="2023-12-31",
)
len(results)

200

A `SearchParams` object holds the same fields. Use it to build a query in
steps, or to reuse one set of filters for more than one call.

In [3]:
params = SearchParams(
    q="single cell",
    db="geo",
    organism="Homo sapiens",
    library_strategy=["RNA-Seq"],
    date_from="2021-01-01",
    date_to="2023-12-31",
)
len(sq.search(params))

200

## Every result

`search` returns one page. `iter_search` yields every result across all pages.

In [4]:
hits = SearchResults(list(itertools.islice(sq.iter_search(params), 100)))
len(hits)

100

## Sort, tabulate, and save

In [5]:
for r in hits.top_cited(5):
    print(r.citation_count, r.accession, "-", r.title)

776 GSE120506 - The Human Testis Cell Atlas via Single-cell RNA-seq (Infant scRNA-seq data set)
602 GSE168004 - Cancer - immune cell interactions drive transitions to mesenchymal-like states in glioblastoma
553 GSE166188 - High throughput joint profiling of chromatin accessibility and protein levels in single cells [PBMC_Stim_Multiome]
532 GSE143706 - Single cell reconstruction of human basal cell diversity in normal and IPF lung (single-cell RNAseq)
373 GSE178325 - Chemical-based external stimulation reprograms human somatic cells into pluripotency (single cell RNA-seq)


In [6]:
df = hits.to_df()
df[["accession", "source", "title", "citation_count"]].head()

,accession,source,title,citation_count
0,GSE172495,geo,SINGLE-CELL CHARACTERIZATION OF A MODEL OF POL...,18
1,GSE155742,geo,Single cell transcriptome profiling of mouse p...,0
2,GSE145809,geo,Single Cell Transcriptomic Characterization of...,36
3,GSE162547,geo,Single-cell-resolved differentiation of human ...,102
4,GSE178325,geo,Chemical-based external stimulation reprograms...,373


In [7]:
hits.to_csv("results.csv")